# Permanent Magnet Materials

Neodymium-Iron-Boron (NdFeB) magnet database and demagnetization curves.

**Contains:**
- MagnetGrade dataclass
- MAGNET_LIBRARY with 25 Arnold Magnetics grades
- MagnetData for temperature-corrected curves

## Executive Summary

**Purpose:** Provide realistic PM properties for motor design

**What it does:** Store and query permanent magnet material data.

### Why It Matters
Motor design requires PM flux linkage and temperature derating curves.

### Quick Start
**Inputs:** Magnet grade (NdFeB, SmCo), operating temperature

**Outputs:** Remanent flux Br, coercivity Hc, temperature derating, J-H curves

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Feeds flux linkage values to 02_pmsm.ipynb and other analysis

**Downstream (used by):** See notebook connections

In [1]:
#| hide
import numpy as np
from dataclasses import dataclass, field

## Theory: NdFeB Magnets

**Strongest permanent magnets available.**

**Properties:**
- Remanence (Br): ≈1.0–1.5 T at 20°C
- Coercivity (Hcb, Hcj): Depends on grade
- Temperature effects: Br and Hcj decline

**Grades:**
- N-series (80°C): Room temperature
- M-series (100°C): Moderate temperature
- H/SH/UH/EH (120–200°C): High temperature

In [2]:
#| export
@dataclass
class MagnetGrade:
    """Room-temperature properties of a permanent magnet grade."""
    grade:     str
    br:        float
    hcb:       float
    hcj:       float
    alpha_br:  float
    alpha_hcj: float

In [3]:
#| export
MAGNET_LIBRARY: dict[str, MagnetGrade] = {
    # N-series (Tmax 80°C)
    "N35":  MagnetGrade("N35",  1.190, 868, 955,  -0.12, -0.60),
    "N38":  MagnetGrade("N38",  1.235, 899, 955,  -0.12, -0.60),
    "N40":  MagnetGrade("N40",  1.275, 923, 955,  -0.12, -0.60),
    "N42":  MagnetGrade("N42",  1.305, 947, 955,  -0.12, -0.60),
    "N45":  MagnetGrade("N45",  1.350, 955, 955,  -0.12, -0.60),
    "N48":  MagnetGrade("N48",  1.400, 955, 955,  -0.12, -0.60),
    "N50":  MagnetGrade("N50",  1.420, 955, 955,  -0.12, -0.60),
    "N52":  MagnetGrade("N52",  1.455, 836, 876,  -0.12, -0.60),
    "N55":  MagnetGrade("N55",  1.485, 836, 876,  -0.12, -0.60),
    # M-series (Tmax 100°C)
    "N35M": MagnetGrade("N35M", 1.190, 868, 1114, -0.12, -0.55),
    "N38M": MagnetGrade("N38M", 1.235, 899, 1114, -0.12, -0.55),
    "N40M": MagnetGrade("N40M", 1.275, 923, 1114, -0.12, -0.55),
    "N42M": MagnetGrade("N42M", 1.305, 947, 1114, -0.12, -0.55),
    "N45M": MagnetGrade("N45M", 1.350, 955, 1114, -0.12, -0.55),
    "N48M": MagnetGrade("N48M", 1.380, 955, 1114, -0.12, -0.55),
    "N50M": MagnetGrade("N50M", 1.410, 955, 1114, -0.12, -0.55),
    # H-series (Tmax 120°C)
    "N30H": MagnetGrade("N30H", 1.110, 836, 1353, -0.12, -0.50),
    "N33H": MagnetGrade("N33H", 1.150, 868, 1353, -0.12, -0.50),
    "N35H": MagnetGrade("N35H", 1.190, 899, 1353, -0.12, -0.50),
    "N38H": MagnetGrade("N38H", 1.235, 923, 1353, -0.12, -0.50),
    "N40H": MagnetGrade("N40H", 1.275, 947, 1353, -0.12, -0.50),
    "N42H": MagnetGrade("N42H", 1.305, 955, 1353, -0.12, -0.50),
    "N45H": MagnetGrade("N45H", 1.350, 955, 1353, -0.12, -0.50),
    "N48H": MagnetGrade("N48H", 1.380, 955, 1194, -0.12, -0.50),
    "N50H": MagnetGrade("N50H", 1.420, 975, 1114, -0.12, -0.50),
}

In [4]:
#| export
@dataclass
class MagnetData:
    """Temperature-corrected demagnetization curves."""
    grade:         str
    from_standard: str
    temperature_C: float
    T_max:         float
    Br:            float
    Hcj:           float
    alpha_br:      float
    alpha_hcj:     float
    
    H:     np.ndarray = field(default_factory=lambda: np.array([]))
    J:     np.ndarray = field(default_factory=lambda: np.array([]))
    B:     np.ndarray = field(default_factory=lambda: np.array([]))
    J_20C: np.ndarray = field(default_factory=lambda: np.array([]))
    B_20C: np.ndarray = field(default_factory=lambda: np.array([]))
    
    def calculate_JH(self) -> None:
        """Compute temperature-corrected J-H and B-H curves."""
        delta_T  = self.temperature_C - 20
        beta_br  = self.alpha_br  * delta_T * 1e-2
        beta_hcj = self.alpha_hcj * delta_T * 1e-2
        Br_temp  = self.Br  * (1 + beta_br)
        Hcj_temp = self.Hcj * (1 + beta_hcj)
        self.H = np.linspace(-self.Hcj, 0, 100)
        self.J, self.B = self._jh_data(self.H, Br_temp, -Hcj_temp)
        self.J_20C, self.B_20C = self._jh_data(self.H, self.Br, -self.Hcj)
    
    def _jh_data(self, H: np.ndarray, Br_temp: float, Hcj_temp: float, k1: float = -6e-5) -> tuple[np.ndarray, np.ndarray]:
        """Exponential J-H model for NdFeB."""
        u0 = 4 * np.pi * 1e-7
        ur = 1.05
        k2 = np.log(Br_temp + (ur - 1) * u0 * Hcj_temp) / k1 - Hcj_temp
        J2 = np.exp(k1 * (k2 + H))
        J_cal = Br_temp + u0 * (ur - 1) * H - J2
        B_cal = Br_temp + u0 * ur * H - J2
        return J_cal, B_cal

## Test

Verify magnet grades and temperature correction.

In [5]:
# Test MAGNET_LIBRARY
assert "N42" in MAGNET_LIBRARY
assert len(MAGNET_LIBRARY) == 25
n42 = MAGNET_LIBRARY["N42"]
assert n42.br == 1.305
print(f"✓ MAGNET_LIBRARY: {len(MAGNET_LIBRARY)} grades")

# Test MagnetData
magnet = MagnetData(
    grade="N42",
    from_standard="Arnold Magnetics",
    temperature_C=80,
    T_max=80,
    Br=n42.br,
    Hcj=n42.hcj * 1e3,
    alpha_br=n42.alpha_br,
    alpha_hcj=n42.alpha_hcj,
)
magnet.calculate_JH()
assert len(magnet.H) == 100
assert len(magnet.B) == 100
print(f"✓ MagnetData.calculate_JH(): {len(magnet.H)} points")
print("\nAll tests passed!")

✓ MAGNET_LIBRARY: 25 grades
✓ MagnetData.calculate_JH(): 100 points

All tests passed!
